# Arabic News Summarizer — AraT5v2 + Arabic XL-Sum

**Primary restartable training notebook.** Choose **Runtime → Change runtime type → T4 GPU**, then **Run all**. Authorize Google Drive when asked. No Hugging Face account or token is required for the public assets.

Defaults run the bounded **full** plan. Set `SUMMARY_MODE=dev` in the configuration cell for a short end-to-end development run. The notebook automatically runs smoke → pretrained baseline → two supervised experiments → validation selection → held-out evaluation. A Colab allocation may end before the plan completes; rerun this notebook to recover.

Persistent Drive checkpoints include weights, optimizer, scheduler, trainer step/epoch and RNG state. Only checkpoints with a completed, verified SHA-256 manifest can be resumed. Steps since the latest saved checkpoint can be lost. Codex cannot reopen a terminated Colab session for you.

**Status:** Project code is tested separately; real AraT5v2/T4 scores remain pending until this notebook actually finishes. XL-Sum is CC BY-NC-SA 4.0; the exact model card does not declare a weight license. See `reports/COMPATIBILITY.md` before redistribution.

In [ ]:
import os
import platform
import subprocess
from pathlib import Path

import torch

print("Python:", platform.python_version(), "PyTorch:", torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError("Enable a T4 GPU using Runtime > Change runtime type, then run all again.")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))
subprocess.run(["nvidia-smi"], check=True)


## Persistent storage and mode

The first run needs enough Drive space (40 GB project cap, with a 7 GB checkpoint reserve by default). Critical artifacts are written directly to Drive. Downloads and prepared data are cached there; temporary package installation and the code checkout live in `/content`.

Use the **same mode and RUN_DIR** on restart. Training settings are locked; only time/storage budgets may be raised. Use a new run directory for changed experiments. Never run two sessions against the same directory.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
MODE = os.environ.get("SUMMARY_MODE", "full")  # "dev" or "full"; no per-experiment edits
if MODE not in {"dev", "full"}:
    raise ValueError("SUMMARY_MODE must be dev or full")
RUN_DIR = Path("/content/drive/MyDrive/ArabicNewsSummarizer") / (MODE + "-v1")
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent run directory:", RUN_DIR)


## Install the exact project version

On the first run, the source commit is frozen on Drive. Restarting restores that commit, even if the repository changes. Before merge this notebook uses the review branch; it remains executable from that branch. Dependencies are pinned where APIs are version-sensitive. No remote dataset Python script is executed.

In [ ]:
import json
import sys

REPOSITORY = "https://github.com/nada-ali-13/Arabic-news-summarizer.git"
CODE_REF = "codex/arabic-summarizer"
REPO_DIR = Path("/content/Arabic-news-summarizer")
if not (REPO_DIR / ".git").exists():
    subprocess.run(["git", "clone", "--branch", CODE_REF, REPOSITORY, str(REPO_DIR)], check=True)
source_lock = RUN_DIR / "source_commit.json"
if source_lock.exists():
    commit = json.loads(source_lock.read_text())["commit"]
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", commit], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "--detach", commit], check=True)
else:
    commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
    temporary = source_lock.with_suffix(".tmp")
    temporary.write_text(json.dumps({"commit": commit, "repository": REPOSITORY}))
    temporary.replace(source_lock)
print("Source commit:", commit)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
sys.path.insert(0, str(REPO_DIR))
sys.path.insert(0, str(REPO_DIR / "src"))
os.chdir(REPO_DIR)
# Record installed versions without environment variables or credentials.
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "installed_packages.txt").write_text(freeze)


## Execute or recover the complete plan

`dev`: two smoke steps and ten development optimizer steps. `full`: two smoke steps and two independent 1,200-step experiments (5e-5 and 1e-4), batch 1 with 16-step accumulation, maximum 1,024 source / 128 target tokens. Early stopping uses validation ROUGE-Lsum, patience 3. The smoke model is excluded from selection.

Evaluation uses the same fixed validation IDs across all candidates. The pretrained baseline is evaluated before real fine-tuning. The final test subset is untouched until the plan finishes. Arabic ROUGE uses Unicode word tokens rather than the default English-only tokenizer. Read the dataset statistics JSON and inspect saved qualitative examples.

The default session budget is 3 hours and the cumulative budget 7 hours; checks run at optimizer-step boundaries. Saving/evaluation may overrun a boundary. A paused plan is recoverable. If the cumulative budget is exhausted, raise `max_hours` below explicitly; experiments do not expand automatically. If full FP16 overflows, start a new run with `mixed_precision="no"`, or use BF16 on a supporting GPU. T4 does not support BF16.

In [ ]:
from arabic_summarizer.config import load_config
from arabic_summarizer.io import atomic_json
from arabic_summarizer.training import run

config = load_config(REPO_DIR / "configs" / (MODE + ".json"))
# Optional budget change on restart, without changing experiment semantics:
# from dataclasses import replace
# config = replace(config, max_hours=10, session_hours=4)
print(json.dumps(config.to_dict(), indent=2, ensure_ascii=False))
result = run(config, RUN_DIR)
print(json.dumps(result, indent=2, ensure_ascii=False))
atomic_json(RUN_DIR / "notebook_last_result.json", result)


## Inspect actual results and qualitative samples

No scores are filled in ahead of execution. `final_report.json` exists only after completed training and final comparison. `best.json` can already point to a completed candidate while later training is paused. Review examples for invented facts, repeated sentences, omissions, fluency and incomplete outputs. The base denoising model may produce poor summaries before supervised transfer learning.

In [ ]:
from IPython.display import display

for name in ["dataset_statistics.json", "experiments.json", "best.json", "final_report.json"]:
    path = RUN_DIR / name
    if path.exists():
        value = json.loads(path.read_text())
        print("\n", name)
        if name == "experiments.json":
            value = {k: {field: row.get(field) for field in ["status", "metrics", "best_checkpoint", "export"]}
                     for k, row in value.items()}
        display(value)
    else:
        print(name, "— pending")
examples_path = RUN_DIR / "best_test.json"
if examples_path.exists():
    examples = json.loads(examples_path.read_text())["examples"][:5]
    for example in examples:
        display({k: example[k] for k in ["id", "reference", "prediction"]})


## Launch the Arabic application (optional)

The best model/tokenizer are independent immutable export directories under `RUN_DIR/exports/`. `best.json` identifies the winner. Copy the selected export to `artifacts/best` for local use, or set `ARABIC_SUMMARIZER_MODEL` to its Drive path. No Hugging Face account is needed. Temporary Gradio sharing exposes the app via a public expiring link; run this cell only when you want to present it. Requests are handled one at a time to protect GPU memory.

In [ ]:
import gc

from app import build_app

best_path = RUN_DIR / "best.json"
if not best_path.exists():
    raise RuntimeError("No completed fine-tuned export yet; resume training first.")
os.environ["ARABIC_SUMMARIZER_MODEL"] = json.loads(best_path.read_text())["path"]
gc.collect()
torch.cuda.empty_cache()

demo = build_app()
demo.launch(share=True, debug=False)


## Recovery checklist

1. Reopen this notebook, select a T4, and run all again. Authorize Drive.
2. Keep the same mode and run path. Frozen source/data/model revisions are restored.
3. Completed experiments are skipped; the newest valid full-state checkpoint resumes automatically.
4. A corrupt newest checkpoint falls back to an older valid one. If none survives, recovery fails clearly and preserves files for inspection. Use a new run directory only if restarting is intentional.
5. An immutable completed export stays available even when later experiments perform worse.

Drive itself is not a transactional filesystem across disconnections. Commit markers and hashes detect partial writes but cannot restore deleted/corrupted backups. Checkpoint SHA verification adds I/O time. Unsaved optimizer steps may be lost. Free Colab GPU availability, connection lifetime and final model quality are not guaranteed.